# **Get data**

In [ ]:
! mkdir ~/.kaggle

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download narayan63/netflix-popular-movies-dataset

In [ ]:
! unzip netflix-popular-movies-dataset

# *Imports*

In [ ]:
import numpy as np
import pandas as pd
import os
import re
import random
import json
import string
import tensorflow as tf
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from pathlib import Path

# Data preparation

In [ ]:
df = pd.read_csv('n_movies.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df = df[['title', 'year', 'genre', 'description', 'stars']]

In [ ]:
df.info()

In [ ]:
df['year'] = df['year'].fillna('')
df['genre'] = df['genre'].fillna('')

In [ ]:
df.info()

# Preprocessing

## Data harmonization

In [ ]:
def preproc_stars(stars):
    stars = re.sub(r"[\[\]']",'', stars)
    stars = stars.split(',')
    stars = [re.sub(r'''["',]''', '', act.strip()) for act in stars]
    stars = ', '.join([act for act in stars if act if stars!=''])
    return f'Stars: {stars}'


def preproc_year(year):
    year = re.findall(r'\d{4}', year)
    try:
        return year[0]
    except IndexError:
        return np.NaN


df['clean_year'] = df['year'].apply(preproc_year)
df = df[df['clean_year'].notna()]
df['clean_stars'] = df['stars'].apply(preproc_stars)
df['clean_title'] = df['title'].apply(lambda title: title.lower())

In [ ]:
prep_df = df[['clean_title', 'clean_stars', 'clean_year', 'genre', 'text']]

## One-hot encodings for "genre"

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

prep_df['genre'] = prep_df['genre'].apply(lambda elem: [gen.strip() for gen in elem.split(',')])

mlb = MultiLabelBinarizer()

df_genres = pd.DataFrame(mlb.fit_transform(prep_df['genre']), columns=mlb.classes_, index=df.index)
prep_df = prep_df.drop('genre', axis=1).join(df_genres)

## Clean texts

In [ ]:
def preproc_desc(text):
    if 'www.' in text or 'http:' in text or 'https:' in text or '.com' in text:
        text = re.sub(r"([^ ]+(?<=\.[a-z]{3}))", "url", text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text.lower()

df['text'] = df['description'].apply(preproc_desc)

In [ ]:
prep_df = prep_df.drop_duplicates(subset=['text']).reset_index(drop=True)

In [ ]:
prep_df['full_text'] = pd.Series([f"{text}. {stars}" for text, stars in zip(prep_df['text'], prep_df['clean_stars'])])

In [ ]:
prep_df = prep_df.drop(['text', 'clean_stars'], axis=1)

In [ ]:
prep_df.info()

## Make input and output columns

In [ ]:
input_columns = [name for name in prep_df.columns if name!='full_text']

prep_df['input'] = prep_df.apply(
          lambda row: ' '.join([f" <{name}> {row[name]}" for name in input_columns]) + ' ' + f"{tokenizer.eos_token}", axis=1)
prep_df['output'] = prep_df.apply(lambda row: f" <full_text> {row['full_text']} {tokenizer.eos_token}", axis=1)

train_data = prep_df[['input', 'output']].copy()

## Load data

In [ ]:
data_path = "/content/"
data_path = Path(data_path)
dataset_path = data_path / 'data.txt'
n = 0
with dataset_path.open('a', encoding='utf-8') as file:
    for input_text, target_text in zip(train_data['input'], train_data['output']):
      file.write(input_text + ' ' + target_text + '\n')
      n+=1
      if n>15:
          break

In [ ]:
print(train_data.columns)

# Create generation model

# Generation model

## Get model

In [ ]:
model_name='gpt2',
cache_dir='model_cache'
output_name='fine_tuned_model'
num_train_epochs=4
per_device_train_batch_size=4
learning_rate=5e-5
save_steps=10_000

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)

## Model training

In [ ]:
train_dataset = TextDataset(
            tokenizer=tokenizer,
            file_path=str(dataset_path),
            block_size=256
            )

data_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer, mlm=False
            )

training_args = TrainingArguments(
            output_dir=str(data_path / output_name),
            overwrite_output_dir=True,
            label_names = ['full_text'],
            num_train_epochs=2,
            per_device_train_batch_size=4,
            save_steps=save_steps,
            learning_rate=learning_rate,
            save_total_limit=2,
            logging_dir=str(data_path / 'logs'),
)

trainer = Trainer(
            model=model,
            args=training_args,
            data_collator=data_collator,
            train_dataset=train_dataset,
)


trainer.train()
model.save_pretrained(str(data_path / output_name))
tokenizer.save_pretrained(str(data_path / output_name))


## Load model

In [ ]:
model_path = Path(data_path) / output_name
tokenizer = GPT2Tokenizer.from_pretrained(str(model_path))
model = GPT2LMHeadModel.from_pretrained(str(model_path))
model.eval()

# Generate text

In [ ]:
def generate_desc(title: str, year: int, genre: list,
                    model=model,
                    max_length=100,
                    num_return_sequences=1,
                    temperature=1.0,
                    top_k=0,
                    top_p=1.0,
                    do_sample=False):

        genres_names = ['Action', 'Adventure', 'Animation',
       'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family',
       'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Music',
       'Musical', 'Mystery', 'News', 'Reality-TV', 'Romance', 'Sci-Fi',
       'Short', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']

        data = {g: 1 if g in genre else 0 for g in genres_names}
        data['clean_title'] = title
        data['clean_year'] = year


        # Формирование prompt
        prompt_text = ''.join([f"{k} {v} " for k, v in data.items()]) + f"{tokenizer.eos_token} <full_text> "

        # Кодирование текста в формате, пригодном для модели
        encoded_input = tokenizer.encode(prompt_text, return_tensors='pt')

        # Генерация текстов
        outputs = model.generate(
            encoded_input,
            max_length=max_length + len(encoded_input[0]),
            num_return_sequences=num_return_sequences,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            no_repeat_ngram_size=2
        )

        # Декодирование результатов
        all_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

        # Удаление входных данных из текстов
        prompt_length = len(tokenizer.decode(encoded_input[0], skip_special_tokens=True))
        trimmed_texts = [text[prompt_length:] for text in all_texts]

        # Возврат результатов в виде словаря
        return trimmed_texts

In [ ]:
unique_title = prep_df['clean_title'].unique().tolist()
unique_year = prep_df['clean_year'].unique().tolist()
unique_genre = df['genre'].unique().tolist()

In [ ]:
unique_genre[0]

## Use input data from dataset to test work of generation model

In [ ]:
title = unique_title[1]
year = unique_year[1]
genre = ['Action', 'Comedy', 'Drama']


generated_texts = generate_desc(
    title=title,
    year=year,
    genre=genre,
    max_length=100,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.95,  # Слегка уменьшаем уверенность
    top_k=10,         # Уменьшаем количество рассматриваемых верхних k слов
    top_p=0.95        # Уменьшаем "ядерность" распределения
)
print(generated_texts[0])

## The generation model has quite good text. To make the text better suited for a particular task, you can try changing the parameters:
## - max_length (maximum length of text)
## - temperature (text sentiment)
## - top_k, top_p (stratigies)

# Thank you for your attention! ✊

---
